# Parts F and G: the test set

This is where test scores are computed and read. Each notebook from 01 to 05 froze its choices on val, then wrote its test predictions without scoring them, so this notebook needs none of their model classes.

1. Scores on test
2. F1: val against test, and the threshold
3. F2: entity never seen in training
4. F3: with and without an explicit keyword
5. F4: without the opinion adjectives
6. F5: 3 seeds, bootstrap and McNemar
7. F6: 30 classified errors
8. G: final table

F2, F3 and F4 use the fine-tuned E1 models.


**Running this notebook.** Each model has its own cell, and a finished run is saved on Drive. If a cell fails, rerun only that cell. After a lost session, run the notebook again: finished runs reload in seconds. To train one again, pass `retrain=True` in its cell.

## Setup

In [ ]:
import os
import sys
from google.colab import drive
drive.mount("/content/drive")

REPO = "AR-aspect-sentiment"

if not os.path.exists(f"/content/{REPO}"):
    !git clone https://github.com/cad0m/{REPO}.git /content/{REPO}
else:
    !git -C /content/{REPO} pull

%cd /content/{REPO}

if f"/content/{REPO}" not in sys.path:
    sys.path.insert(0, f"/content/{REPO}")

!pip install -q arabert emoji gensim sentencepiece tashaphyne

OUTPUT_DIR = f"/content/drive/MyDrive/{REPO}"
os.environ["OUTPUT_DIR"] = OUTPUT_DIR
!mkdir -p {OUTPUT_DIR} && cp -rn results {OUTPUT_DIR}/

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from arabert.preprocess import ArabertPreprocessor
from torch import nn
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, get_linear_schedule_with_warmup

from src.aspect_names import OPINION_WORDS, aspect_name
from src.data import POLARITIES, category_list, category_matrix, load_test, load_train_val, pair_units, pairs
from src.evaluation import (
    THRESHOLD_GRID,
    accuracy_by_mixed_sentence,
    best_global_threshold,
    bootstrap_interval,
    category_report,
    category_scores,
    mcnemar_test,
    pipeline_outcomes,
    pipeline_scores,
    polarity_f1_by_class,
    polarity_scores,
    predict_labels,
)
from src.plots import (
    plot_category_f1,
    plot_category_gain,
    plot_f1_by_threshold,
    plot_grouped_bars,
    plot_mixed_accuracy,
    plot_model_comparison,
    plot_pipeline_outcomes,
    plot_polarity_confusion,
    plot_polarity_f1,
    plot_quality_against_cost,
)
from src.preprocessing import NORM_STEPS, build_pipeline
from src.tracking import RESULTS_DIR, cached_run, load_scores, log_results, result_row
from src.training import DEVICE, collect_outputs, compact_state, fit, set_seed, token_inputs, tokenized_rows

SEED = 42
BERT_NAMES = {
    "arabert": "aubmindlab/bert-base-arabertv02",
    "camelbert": "CAMeL-Lab/bert-base-arabic-camelbert-msa",
}
MAX_LENGTH = 128
BATCH_SIZE = 32
TEST_RESULTS_PATH = RESULTS_DIR / "test_results.csv"
sns.set_theme(style="whitegrid", context="notebook")
print("Device:", DEVICE)

In [ ]:
opinions, train, val = load_train_val(SEED)
classes = category_list(opinions)
category_ids = {name: index for index, name in enumerate(classes)}
polarity_ids = {name: index for index, name in enumerate(POLARITIES)}

val_labels = category_matrix(val.categories, classes)
train_units = pair_units(pairs(opinions, train))
train_units = train_units[~train_units.ambiguous].reset_index(drop=True)
val_units = pair_units(pairs(opinions, val))

test_opinions, test = load_test()
test_labels = category_matrix(test.categories, classes)
test_pairs = pairs(test_opinions, test)
test_units = pair_units(test_pairs)
clear_units = ~test_units.ambiguous.to_numpy(dtype=bool)

print(f"{len(test)} test sentences, {len(test_units)} (sentence, category) pairs")

## 1. Scores on test

The score files hold one row per seed. Models with one run have a single row.

In [ ]:
FAMILIES = {
    "TF-IDF + LR": "tfidf_lr",
    "zero-shot": "zero_shot",
    "neural": "neural",
    "frozen BERT": "frozen_bert",
    "fine-tuned BERT": "fine_tuned",
    "multi-task BERT": "multitask",
}

category_runs = {name: load_scores(f"{run}_category", "test", test.sentence_id) for name, run in FAMILIES.items()}
polarity_runs = {name: load_scores(f"{run}_polarity", "test") for name, run in FAMILIES.items()}
polarity_runs["sentence sentiment copied"] = load_scores("camelbert_sentiment_polarity", "test")


def seed_predictions(name):
    run = category_runs[name]
    return [predict_labels(scores, run["meta"]["thresholds"]) for scores in run["scores"]]


def top_polarity(scores):
    return np.array(POLARITIES, dtype=object)[scores.argmax(axis=1)]


def seed_polarities(name):
    return [top_polarity(scores) for scores in polarity_runs[name]["scores"]]


def pipeline_result(name):
    run = load_scores(f"{FAMILIES[name]}_pipeline", "test")
    return pipeline_outcomes(test_pairs, pd.DataFrame(run["row_ids"]).assign(polarity=top_polarity(run["scores"][0])))

In [ ]:
test_rows = []
for name in category_runs:
    test_rows += [result_row("category", name, seed=index, **category_scores(test_labels, predictions)) for index, predictions in enumerate(seed_predictions(name))]
    test_rows.append(result_row("pipeline", name, micro_f1=pipeline_scores(pipeline_result(name))["pipeline_f1"]))
for name in polarity_runs:
    test_rows += [result_row("polarity", name, seed=index, **polarity_scores(test_units, predicted)) for index, predicted in enumerate(seed_polarities(name))]

test_table = log_results(test_rows, path=TEST_RESULTS_PATH)
mean_scores = test_table.groupby(["task", "model"], as_index=False)[["micro_f1", "macro_f1"]].mean().rename(columns={"model": "name"})

plot_model_comparison(mean_scores[mean_scores.task == "category"], "micro_f1", title="Categories on test (official metric)")
plot_model_comparison(mean_scores[mean_scores.task == "polarity"], "macro_f1", title="Polarity on test")
plot_model_comparison(mean_scores[mean_scores.task == "pipeline"], "micro_f1", title="Full pipeline on test")

The retained model is fixed by its val score, not by its test score:

In [ ]:
def val_micro_f1(name):
    run = load_scores(f"{FAMILIES[name]}_category", "val", val.sentence_id)
    return category_scores(val_labels, predict_labels(run["scores"], run["meta"]["thresholds"]))["micro_f1"]


val_ranking = sorted(FAMILIES, key=val_micro_f1, reverse=True)
final_name, second_name = val_ranking[:2]
print("Retained model:", final_name, "| second:", second_name)

final_report = category_report(test_labels, seed_predictions(final_name)[0], classes)
plot_category_f1(final_report, title=f"F1 per category on test ({final_name})")
plot_category_gain(category_report(test_labels, seed_predictions("TF-IDF + LR")[0], classes), final_report, "TF-IDF + LR", final_name)
print(f"{(final_report.predicted == 0).sum()} categories are never predicted")

## 2. F1: val against test, and the threshold (Q10)

In [ ]:
def category_shares(sentence_table):
    return sentence_table.categories.explode().dropna().value_counts(normalize=True).reindex(classes).fillna(0)


val_shares = category_shares(val)
test_shares = category_shares(test)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(val_shares, test_shares, color="#2E6DA4", s=45, edgecolor="white")
largest = max(val_shares.max(), test_shares.max()) * 1.05
ax.plot([0, largest], [0, largest], color="black", linestyle="--", linewidth=1)
for category in (val_shares - test_shares).abs().nlargest(4).index:
    ax.annotate(category, (val_shares[category], test_shares[category]), xytext=(5, -10), textcoords="offset points", fontsize=8)
ax.set_xlabel("share of opinions in val")
ax.set_ylabel("share of opinions in test")
ax.set_title("Category distribution: points on the line did not move")
plt.tight_layout()
plt.show()

plot_grouped_bars(pd.DataFrame({
    "val": val_units.polarity.value_counts(normalize=True),
    "test": test_units.polarity.value_counts(normalize=True),
}).T[POLARITIES], "share", "Polarity distribution")

In [ ]:
final_val_scores = load_scores(f"{FAMILIES[final_name]}_category", "val", val.sentence_id)["scores"]
final_test_scores = category_runs[final_name]["scores"][0]

val_threshold, _ = best_global_threshold(final_val_scores, val_labels)
posthoc_threshold, test_curve = best_global_threshold(final_test_scores, test_labels)
score_at_val_threshold = category_scores(test_labels, predict_labels(final_test_scores, val_threshold))["micro_f1"]

plot_f1_by_threshold(THRESHOLD_GRID, test_curve, val_threshold, title=f"Micro-F1 on test by threshold ({final_name}): analysis only", posthoc_threshold=posthoc_threshold)
print(f"Threshold from val {val_threshold:.2f}: micro-F1 {score_at_val_threshold:.4f}")
print(f"Best afterwards {posthoc_threshold:.2f}: micro-F1 {test_curve.max():.4f}")

**Conclusions (Q10)**

_To fill in after the run: shift between val and test, gap between the chosen threshold and the best one, and what to set up in production._

## 3. F2: entity never seen in training (Q11)

The E1 polarity model is trained again without any `FOOD_DRINKS` pair, then tested on the `FOOD_DRINKS` pairs of test. It still reads the name of the unseen category, thanks to the pair input.

In [ ]:
fine_tuned_meta = category_runs["fine-tuned BERT"]["meta"]
polarity_backbone = fine_tuned_meta["polarity_backbone"]
category_backbone = fine_tuned_meta["backbone"]

tokenizers = {name: AutoTokenizer.from_pretrained(BERT_NAMES[name]) for name in {polarity_backbone, category_backbone}}
PREPARE_TEXT = {"arabert": ArabertPreprocessor(BERT_NAMES["arabert"]).preprocess, "camelbert": lambda text: text}


def make_loader(rows, backbone, shuffle=False):
    return DataLoader(rows, batch_size=BATCH_SIZE, shuffle=shuffle, collate_fn=DataCollatorWithPadding(tokenizers[backbone]))


def pair_rows(pair_table):
    texts = pair_table.text.map(PREPARE_TEXT[polarity_backbone])
    category_names = [aspect_name(category, "arabic") for category in pair_table.category]
    labels = pair_table.polarity.map(polarity_ids).fillna(-1).astype(int).tolist()
    return tokenized_rows(tokenizers[polarity_backbone], texts, category_names, labels, MAX_LENGTH)


def new_polarity_model():
    return AutoModelForSequenceClassification.from_pretrained(BERT_NAMES[polarity_backbone], num_labels=len(POLARITIES)).to(DEVICE)


def predict_polarities(model, pair_table):
    outputs = collect_outputs(make_loader(pair_rows(pair_table), polarity_backbone), lambda batch: model(**token_inputs(batch)).logits)
    return top_polarity(outputs.numpy())


def is_food_drinks(pair_table):
    return pair_table.category.str.startswith("FOOD_DRINKS")

In [ ]:
def train_without_food_drinks():
    seen_units = train_units[~is_food_drinks(train_units)].reset_index(drop=True)
    seen_val_units = val_units[~is_food_drinks(val_units)].reset_index(drop=True)

    counts = seen_units.polarity.map(polarity_ids).value_counts().sort_index().to_numpy()
    loss_function = nn.CrossEntropyLoss(weight=torch.tensor(len(seen_units) / (len(POLARITIES) * counts), dtype=torch.float32, device=DEVICE))

    set_seed(SEED)
    model = new_polarity_model()
    train_loader = make_loader(pair_rows(seen_units), polarity_backbone, shuffle=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
    total_steps = 4 * len(train_loader)
    scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
    history = fit(
        model, train_loader,
        lambda model, batch: loss_function(model(**token_inputs(batch)).logits, batch["labels"]),
        lambda model: polarity_scores(seen_val_units, predict_polarities(model, seen_val_units))["macro_f1"],
        optimizer, scheduler, epochs=4, patience=2, mixed_precision=True,
    )
    return {"state": compact_state(model), "history": history}


unseen_run = cached_run("f2_without_food_drinks", train_without_food_drinks)

In [ ]:
unseen_model = new_polarity_model()
unseen_model.load_state_dict(unseen_run["state"])
unseen_model.eval()

without_entity = cached_run("f2_predictions", lambda: predict_polarities(unseen_model, test_units))
with_entity = seed_polarities("fine-tuned BERT")[0]
food_drinks_units = is_food_drinks(test_units).to_numpy()

plot_grouped_bars(pd.DataFrame({
    "FOOD_DRINKS, in training": polarity_scores(test_units[food_drinks_units], with_entity[food_drinks_units]),
    "FOOD_DRINKS, never seen": polarity_scores(test_units[food_drinks_units], without_entity[food_drinks_units]),
    "other entities, full model": polarity_scores(test_units[~food_drinks_units], with_entity[~food_drinks_units]),
    "other entities, without FOOD_DRINKS": polarity_scores(test_units[~food_drinks_units], without_entity[~food_drinks_units]),
}).T, "score", "Polarity with and without the entity in training")

Sentences with opposite polarities, on every model:

In [ ]:
plot_mixed_accuracy({name: accuracy_by_mixed_sentence(test_units, seed_polarities(name)[0]) for name in polarity_runs})
plot_polarity_f1({name: polarity_f1_by_class(test_units, seed_polarities(name)[0]) for name in polarity_runs})

**Conclusions (Q11)**

_To fill in after the run: what the model does with an entity it never saw, and what this means for another domain such as restaurants._

## 4. F3: with and without an explicit keyword

Each gold pair is split by the presence of a keyword of its entity in the sentence.

In [ ]:
ENTITY_KEYWORDS = {
    "HOTEL": ["فندق"],
    "ROOMS": ["غرف"],
    "ROOMS_AMENITIES": ["حمام", "سرير", "تكييف", "تلفزيون", "اثاث"],
    "FACILITIES": ["مسبح", "مرافق", "نادي", "موقف", "مصعد"],
    "SERVICE": ["خدم", "موظف", "استقبال", "طاقم"],
    "FOOD_DRINKS": ["افطار", "فطور", "طعام", "اكل", "مطعم"],
    "LOCATION": ["موقع", "مكان", "قريب"],
}
normalize = build_pipeline(NORM_STEPS)
row_of_sentence = dict(zip(test.sentence_id, range(len(test))))

e1_predictions = seed_predictions("fine-tuned BERT")[0]
sentence_of_pair = test_units.sentence_id.map(row_of_sentence).to_numpy()
column_of_pair = test_units.category.map(category_ids).to_numpy()

keyword_check = test_units.assign(
    has_keyword=[
        any(keyword in normalize(row.text) for keyword in ENTITY_KEYWORDS[row.category.split("#")[0]])
        for row in test_units.itertuples()
    ],
    detected=e1_predictions[sentence_of_pair, column_of_pair] == 1,
    polarity_correct=with_entity == test_units.polarity.to_numpy(dtype=object),
)
keyword_table = keyword_check[~keyword_check.ambiguous].groupby("has_keyword")[["detected", "polarity_correct"]].mean()
keyword_table.index = keyword_table.index.map({True: "keyword present", False: "no keyword"})
keyword_table.columns = ["category found (recall)", "polarity correct"]

print(keyword_check.has_keyword.value_counts().to_string())
plot_grouped_bars(keyword_table, "share", "Gold pairs with and without an explicit keyword")

**Conclusions (F3)**

_To fill in after the run: how far the model depends on the surface word, and what it says about the shortcut of Q2._

## 5. F4: without the opinion adjectives

Take the decisions that are right, remove the opinion adjectives from those sentences, and see how many stay right.

In [ ]:
opinion_words = {normalize(word) for word in OPINION_WORDS}


def has_opinion_word(text):
    return any(normalize(word) in opinion_words for word in text.split())


def without_opinion_words(text):
    return " ".join(word for word in text.split() if normalize(word) not in opinion_words)


def load_e1(task, backbone, n_outputs):
    model = AutoModelForSequenceClassification.from_pretrained(BERT_NAMES[backbone], num_labels=n_outputs).to(DEVICE)
    model.load_state_dict(cached_run(f"e1_{task}_{backbone}_seed{SEED}", None)["state"])
    return model.eval()


right_pairs = test_units[(~test_units.ambiguous) & (with_entity == test_units.polarity.to_numpy(dtype=object))]
right_pairs = right_pairs[right_pairs.text.map(has_opinion_word)].reset_index(drop=True)
stripped_pairs = right_pairs.assign(text=right_pairs.text.map(without_opinion_words))

polarities_after = cached_run("f4_polarities", lambda: predict_polarities(load_e1("polarity", polarity_backbone, len(POLARITIES)), stripped_pairs))
polarity_kept = (polarities_after == right_pairs.polarity.to_numpy(dtype=object)).mean()

In [ ]:
def stripped_category_scores(texts):
    model = load_e1("category", category_backbone, len(classes))
    prepared = texts.map(without_opinion_words).map(PREPARE_TEXT[category_backbone])
    loader = make_loader(tokenized_rows(tokenizers[category_backbone], prepared, max_length=MAX_LENGTH), category_backbone)
    return collect_outputs(loader, lambda batch: model(**token_inputs(batch)).logits).sigmoid().numpy()


with_opinion = test.text.map(has_opinion_word).to_numpy()
scores_after = cached_run("f4_categories", lambda: stripped_category_scores(test.text[with_opinion]))
labels_kept = test_labels[with_opinion]
found_before = (e1_predictions[with_opinion] * labels_kept) == 1
found_after = (predict_labels(scores_after, fine_tuned_meta["thresholds"]) * labels_kept) == 1

plot_model_comparison(pd.DataFrame({
    "name": ["polarity decisions kept", "category decisions kept"],
    "share": [polarity_kept, (found_before & found_after).sum() / found_before.sum()],
}), "share", title="Right decisions that survive without the opinion words")

**Conclusions (F4)**

_To fill in after the run: the drop on each task. Polarity should fall much more than category detection._

## 6. F5: 3 seeds, bootstrap and McNemar

The bootstrap draws whole reviews, 1,000 times, on the first seed. McNemar compares the two best models on each (sentence, category) decision.

In [ ]:
def interval_row(name):
    predictions = seed_predictions(name)
    seed_values = [category_scores(test_labels, seed_prediction)["micro_f1"] for seed_prediction in predictions]
    lower, upper = bootstrap_interval(lambda rows: category_scores(test_labels[rows], predictions[0][rows])["micro_f1"], test.review_id)
    return {"name": name, "first_seed": seed_values[0], "seed_mean": np.mean(seed_values), "seed_std": np.std(seed_values), "lower": lower, "upper": upper}


interval_table = pd.DataFrame([interval_row(name) for name in val_ranking]).sort_values("first_seed")

fig, ax = plt.subplots(figsize=(8, 0.5 * len(interval_table) + 1.5))
ax.errorbar(interval_table.first_seed, interval_table.name, capsize=4, marker="o", color="#2E6DA4", linestyle="none",
            xerr=[(interval_table.first_seed - interval_table.lower).clip(lower=0), (interval_table.upper - interval_table.first_seed).clip(lower=0)])
ax.set_xlabel("micro-F1 on test, with 95% bootstrap interval")
ax.set_title("Overlapping intervals mean the gap may be noise")
plt.tight_layout()
plt.show()

interval_table.round(4)

In [ ]:
mcnemar_test(
    (seed_predictions(final_name)[0] == test_labels).ravel(),
    (seed_predictions(second_name)[0] == test_labels).ravel(),
)

**Conclusions (F5)**

_To fill in after the run: width of the intervals, and whether the gap between the two best models is significant._

## 7. F6: 30 classified errors (Q12)

Errors of the E1 pipeline, 10 of each kind, with a class suggested by simple rules.

**This step needs you.** Annotation noise can only be judged by reading the sentence, so read the table and write your corrections in `MANUAL_TYPES`, as `{row: "type"}`.

In [ ]:
outcomes = pipeline_result("fine-tuned BERT")
plot_pipeline_outcomes(outcomes, title="What happens to each (sentence, category) on test")

sentence_text = test.set_index("sentence_id").text
gold_categories = test.set_index("sentence_id").categories
mixed_sentences = test_pairs.groupby("sentence_id").polarity.agg(lambda polarities: {"positive", "negative"} <= set(polarities))
implicit_pairs = set(zip(*test_pairs.loc[test_pairs.target == "NULL", ["sentence_id", "category"]].to_numpy().T))


def is_neighbour(category, others):
    entity, attribute = category.split("#")
    return any(other != category and (other.startswith(entity + "#") or other.endswith("#" + attribute)) for other in others)


def suggest_error_type(row):
    if "neutral" in row.gold or "neutral" in row.predicted:
        return "neutral"
    if mixed_sentences.get(row.sentence_id, False):
        return "mixed polarity"
    if (row.sentence_id, row.category) in implicit_pairs:
        return "implicit aspect"
    if is_neighbour(row.category, gold_categories[row.sentence_id]):
        return "neighbouring category"
    return "annotation noise?"


error_sample = outcomes[outcomes.status != "correct"].groupby("status").sample(10, random_state=SEED).reset_index(drop=True)
error_sample["text"] = error_sample.sentence_id.map(sentence_text)
error_sample["error_type"] = [suggest_error_type(row) for row in error_sample.itertuples()]

pd.set_option("display.max_colwidth", None)
error_sample[["status", "category", "gold", "predicted", "error_type", "text"]]

In [ ]:
MANUAL_TYPES = {}

error_types = error_sample.error_type.copy()
for row, error_type in MANUAL_TYPES.items():
    error_types[row] = error_type

type_shares = error_types.value_counts(normalize=True).rename_axis("name").reset_index(name="share")
plot_model_comparison(type_shares, "share", title="Kinds of error, out of 30")

**Conclusions (Q12)**

_To fill in after the run and after reading the 30 errors: dominant classes, and the share of annotation noise._

## 8. G: final table

In [ ]:
pipeline_by_model = test_table[test_table.task == "pipeline"].set_index("model").micro_f1
polarity_by_model = test_table[test_table.task == "polarity"].groupby("model").macro_f1.mean()


def final_row(name):
    values = [category_scores(test_labels, predictions)["micro_f1"] for predictions in seed_predictions(name)]
    meta = category_runs[name]["meta"]
    return {
        "name": name,
        "category_micro_f1": np.mean(values),
        "seed_std": np.std(values),
        "polarity_macro_f1": polarity_by_model[name],
        "pipeline_f1": pipeline_by_model[name],
        "train_s": meta["train_s"],
        "latency_ms": meta["latency_ms"],
        "size_mb": meta["size_mb"],
    }


final_table = pd.DataFrame([final_row(name) for name in FAMILIES])
final_table.to_csv(RESULTS_DIR / "final_table.csv", index=False)
final_table.round(4)

In [ ]:
plot_quality_against_cost(final_table, "latency_ms", "category_micro_f1", title="Quality against CPU latency")
plot_quality_against_cost(final_table, "size_mb", "category_micro_f1", title="Quality against model size")
plot_quality_against_cost(final_table[final_table.train_s > 0], "train_s", "category_micro_f1", title="Quality against training time")

Confusion matrix of the retained model:

In [ ]:
final_polarities = seed_polarities(final_name)[0]
plot_polarity_confusion(test_units.polarity[clear_units], final_polarities[clear_units], title=f"Polarity confusion on test ({final_name})")

**Conclusions (Q12)**

_To fill in after the run: which model goes to production, and why, with numbers on quality, robustness, cost and latency._